# 02 — Modelagem preditiva da produtividade da soja no RS (N2)

**Disciplina:** Inteligência Artificial — 7ºJ SI — Noite  
**Instituição:** FCI / Universidade Presbiteriana Mackenzie  
**Professor:** Prof. Dr. Leandro Zerbinatti  
**Integrantes:**  
- Gabriel Bello — RA 10416808 — 10416808@mackenzista.com.br  
- Pedro Giovannini Andrade — RA 10425498 — 10425498@mackenzista.com.br  

**Síntese:** Carrega a base N2 (PAM/IBGE + ONI/NOAA + NASA POWER), treina Ridge, Random Forest e Gradient Boosting com validação temporal e compara com linhas de base (média histórica e safra anterior). Gera métricas (MAE, RMSE, R²), importância de atributos e indicador de risco de quebra.

**Histórico:**
| Data | Autor | Descrição |
|---|---|---|
| 2026-11-16 | Pedro Giovannini Andrade | Versão inicial (N2). |
| 2026-11-16 | Gabriel Bello | Revisão de gráficos e seção de risco. |

## 1. Imports e caminhos

In [ ]:
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import (mean_absolute_error, mean_squared_error,
                             r2_score, confusion_matrix, classification_report)

REPO  = os.path.abspath(os.path.join(os.getcwd(), '..'))
DADOS = os.path.join(REPO, 'dados')
FIG   = os.path.join(REPO, 'figuras')

## 2. Base integrada (PAM/IBGE + ONI/NOAA + NASA POWER)

A base N2 contém 1.197 pares município-safra após integração das três fontes (Top 50 municípios produtores do RS, 2000–2023).

In [ ]:
base = pd.read_csv(os.path.join(DADOS, 'base_soja_rs_N2.csv'))
base = base.dropna(subset=['rendimento_kg_ha'])
base.shape, base['safra'].min(), base['safra'].max()

In [ ]:
base[['pr_acum_safra','pr_acum_df','dias_calor_extremo',
      'rad_acum','rh_media','gdd','rendimento_kg_ha']].describe().round(2)

## 3. Divisão temporal

Treino: safras ≤ 2018 (897 obs.)   •   Teste: 2019–2023 (250 obs.).

In [ ]:
FEAT_NUM = ['media_hist_mun','rendimento_anterior',
            'oni_medio_safra','oni_min_safra','oni_max_safra',
            'pr_acum_safra','pr_acum_df','dias_chuva','max_dias_secos',
            'tmed_safra','tmed_df','tmax_df_media','dias_calor_extremo',
            'rad_acum','rh_media','gdd']
FEAT_CAT = ['fase_dominante_safra']
TARGET   = 'rendimento_kg_ha'

train = base[base['safra'] <= 2018].dropna(subset=FEAT_NUM+[TARGET])
test  = base[base['safra'] >  2018].dropna(subset=FEAT_NUM+[TARGET])
len(train), len(test)

## 4. Pipeline e modelos

In [ ]:
prep = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')),
                       ('sc',  StandardScaler())]), FEAT_NUM),
    ('cat', OneHotEncoder(handle_unknown='ignore'), FEAT_CAT),
])
MODELOS = {
    'Ridge':            Ridge(alpha=1.0, random_state=0),
    'RandomForest':     RandomForestRegressor(n_estimators=400, max_depth=14,
                                              min_samples_leaf=3, n_jobs=-1,
                                              random_state=0),
    'GradientBoosting': GradientBoostingRegressor(n_estimators=400, max_depth=3,
                                                  learning_rate=0.05,
                                                  random_state=0),
}

def mt(y, yhat):
    return dict(MAE=mean_absolute_error(y,yhat),
                RMSE=np.sqrt(mean_squared_error(y,yhat)),
                R2=r2_score(y,yhat))

Xtr,ytr = train[FEAT_NUM+FEAT_CAT], train[TARGET]
Xte,yte = test [FEAT_NUM+FEAT_CAT], test [TARGET]

res = {'baseline_media_hist':   mt(yte, test['media_hist_mun'].fillna(ytr.mean())),
       'baseline_ano_anterior': mt(yte, test['rendimento_anterior'].fillna(ytr.mean()))}
pred = {}
for n,m in MODELOS.items():
    p = Pipeline([('prep',prep),('mdl',m)]).fit(Xtr,ytr)
    pred[n] = p.predict(Xte)
    res[n]  = mt(yte, pred[n])
pd.DataFrame(res).T.round(2)

## 5. Resultados chave

- Gradient Boosting: **MAE ≈ 598 kg/ha**, **R² ≈ 0,47** — melhor desempenho.
- Supera em ~34% a baseline da média histórica (MAE=908) e ~50% a safra anterior.
- Atributo mais importante no RF: **precipitação acumulada em dez–fev** (janela reprodutiva da soja).

In [ ]:
from IPython.display import Image
Image(os.path.join(FIG, '07_comparacao_modelos.png'))

In [ ]:
Image(os.path.join(FIG, '10_importancia_rf.png'))

In [ ]:
Image(os.path.join(FIG, '08_obs_vs_previsto.png'))

## 6. Indicador de risco de quebra

Risco = rendimento previsto < média histórica municipal. **Recall = 97%** (quase todas as quebras são sinalizadas), **acurácia = 78%**, com baixa taxa de alarme-falso significativo.

In [ ]:
Image(os.path.join(FIG, '11_matriz_risco.png'))